# Base Model Failure Demo — ICD-10 Code Prediction

**Runtime required: A100 GPU** (`Runtime → Change runtime type → A100 GPU`)

This notebook demonstrates **where a general-purpose language model fails on a specialized clinical task** — ICD-10 code prediction.

We load **Qwen2.5-14B-Instruct** — a strong, fully open-source 14B parameter model — and ask it 6 ICD coding questions. No fine-tuning. No RAG. Just the base model doing its best on a domain it wasn't specifically trained for.

This is the problem we solve in the next notebook with LoRA fine-tuning.

**What we do here:**
1. Load Qwen2.5-14B-Instruct in bfloat16 (~28GB VRAM)
2. Define 6 ICD coding questions — code lookup and code prediction
3. Run the base model on all 6
4. Show exactly where and why it fails

## Step 0 — Install Dependencies

⚠️ After running this cell, **restart the runtime** (`Runtime → Restart session`), then continue from Step 1.

In [ ]:
!pip install -q transformers==4.45.0
!pip install -q accelerate==0.34.2

print("Done. Restart the runtime before running any further cells.")

## Step 1 — Verify GPU

Confirm you have an A100. Loading Qwen2.5-14B in bfloat16 requires ~28GB VRAM.

In [ ]:
import torch

print(f"GPU available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:       {torch.cuda.get_device_name(0)}")
    print(f"VRAM total:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"PyTorch:        {torch.__version__}")
else:
    print("No GPU found. Switch runtime to A100 and restart.")

## Step 2 — Define the 6 Test Questions

We lock these in **before** loading any model. Two task types:

- **Code lookup** (Q1–Q3): given an ICD-10 code → describe the condition precisely
- **Code prediction** (Q4–Q6): given a clinical scenario → return the correct code

These are the same 6 questions we'll re-run after LoRA fine-tuning in the next notebook.

In [ ]:
TEST_QUESTIONS = [
    # --- Code lookup ---
    {
        "id": "Q1",
        "type": "lookup",
        "question": "What condition does ICD-10 code J18.9 describe?",
        "expected": "J18.9 — Pneumonia, unspecified organism",
    },
    {
        "id": "Q2",
        "type": "lookup",
        "question": "What is the difference between ICD-10 codes J18.0 and J18.9?",
        "expected": "J18.0 = Bronchopneumonia (patchy, bilateral). J18.9 = Pneumonia unspecified organism (catch-all when no pattern or organism specified).",
    },
    {
        "id": "Q3",
        "type": "lookup",
        "question": "What does ICD-10 code I50.32 mean?",
        "expected": "I50.32 — Chronic diastolic (left ventricular) heart failure",
    },
    # --- Code prediction ---
    {
        "id": "Q4",
        "type": "prediction",
        "question": (
            "A patient presents with productive cough, fever of 38.9°C, and chest X-ray "
            "showing lobar consolidation. No organism was identified on cultures. "
            "What is the most likely ICD-10 code?"
        ),
        "expected": "J18.1 — Lobar pneumonia, unspecified organism (lobar consolidation pattern, no organism identified)",
    },
    {
        "id": "Q5",
        "type": "prediction",
        "question": (
            "What ICD-10 code would you assign to a patient diagnosed with type 2 diabetes "
            "with diabetic chronic kidney disease, stage 3?"
        ),
        "expected": "E11.22 — Type 2 diabetes mellitus with diabetic chronic kidney disease, stage 3 (combination code; do not code N18.3 separately)",
    },
    {
        "id": "Q6",
        "type": "prediction",
        "question": (
            "A physician's note reads: 'Pt admitted with ACS, found to have NSTEMI on troponin trend. "
            "Managed medically, no PCI performed.' What is the primary ICD-10 code?"
        ),
        "expected": "I21.4 — Non-ST elevation (NSTEMI) myocardial infarction",
    },
]

print(f"Locked in {len(TEST_QUESTIONS)} test questions.")
for q in TEST_QUESTIONS:
    print(f"  [{q['id']}] ({q['type']:10s}) {q['question'][:70]}...")

## Step 3 — Load Qwen2.5-14B-Instruct (bfloat16)

We load the **base model with no quantization** — full bfloat16 precision. This is the strongest representation of the model before any domain adaptation.

- 14 billion parameters
- ~28GB VRAM on A100
- Instruction-tuned by Alibaba — knows how to follow prompts
- No API key, no access request, fully open-source

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Loading model in bfloat16 (no quantization)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"\nModel loaded. VRAM used: {vram_used:.1f} GB")

## Step 4 — Run the Base Model on All 6 Questions

Greedy decoding (no sampling) for deterministic, reproducible answers.
The system prompt positions the model as a clinical coding specialist — this is the best prompt we can give without any fine-tuning.

Watch where it hedges, confuses codes, or hallucinates qualifiers.

In [ ]:
import time

SYSTEM_PROMPT = (
    "You are a clinical coding specialist with deep expertise in ICD-10-CM. "
    "Provide accurate, precise ICD-10 code assignments and descriptions. "
    "When predicting a code, state the code first, then the full description, "
    "then a brief clinical rationale."
)

def generate_answer(question, max_new_tokens=300):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - start

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
    tokens = len(generated)
    return answer, elapsed, tokens


print("Running base model on all 6 test questions...\n")
print("=" * 70)

baseline_results = {}
for q in TEST_QUESTIONS:
    print(f"[{q['id']}] ({q['type']}) {q['question']}")
    answer, elapsed, tokens = generate_answer(q["question"])
    baseline_results[q["id"]] = answer
    print(f"\nAnswer:\n{answer}")
    print(f"\nExpected: {q['expected']}")
    print(f"Speed: {tokens/elapsed:.0f} tok/s | Time: {elapsed:.1f}s")
    print("-" * 70)

## Step 5 — Failure Analysis

Score each answer manually: **correct**, **partial**, or **wrong**.

Common failure patterns to look for:
- **Hedging** — correct condition but refuses to commit to a specific code
- **Wrong subcode** — gets the parent code right (e.g. J18) but picks the wrong 4th/5th character
- **Hallucinated qualifiers** — invents details not present in the code definition (e.g. confuses systolic/diastolic)
- **Missing combination codes** — codes two conditions separately instead of using the ICD-10 combination code
- **Abbreviation confusion** — misreads clinical shorthand (ACS, NSTEMI, SOB)

In [ ]:
# Fill in your assessment after reviewing the outputs above
# Options: "correct", "partial", "wrong"
scores = {
    "Q1": "",   # J18.9 — easy, common code
    "Q2": "",   # J18.0 vs J18.9 distinction
    "Q3": "",   # I50.32 — systolic vs diastolic trap
    "Q4": "",   # lobar consolidation → J18.1 (not J18.9)
    "Q5": "",   # combination code E11.22
    "Q6": "",   # NSTEMI abbreviation → I21.4
}

print("Base model scorecard:")
print("=" * 50)
for q in TEST_QUESTIONS:
    qid = q['id']
    score = scores[qid] if scores[qid] else "(not scored)"
    print(f"  [{qid}] {q['type']:10s}  {score:12s}  {q['question'][:50]}...")

scored = [v for v in scores.values() if v]
if scored:
    correct = scored.count('correct')
    partial = scored.count('partial')
    wrong   = scored.count('wrong')
    print(f"\nResult: {correct} correct / {partial} partial / {wrong} wrong out of {len(scored)} scored")

## What's Next

**What we showed:**
- Qwen2.5-14B-Instruct is a strong general model — it handles the easy, well-known codes reasonably well
- It struggles with: subcategory precision, combination codes, clinical abbreviations, and cases where the distinction matters most
- These are exactly the failure modes that matter in production — a billing error is a billing error regardless of how confident the model sounded

**Why RAG won't fix this:**
ICD coding is not a retrieval problem. You can't retrieve your way to understanding that `E11.22` is the combination code for T2DM + CKD stage 3, or that `I50.32` is *diastolic* not systolic. The model needs to have internalized the structure and logic of the ICD-10 code set.

**Next notebook — LoRA fine-tuning:**
We build an instruction dataset from the official CMS ICD-10-CM code set, fine-tune with QLoRA (training <1% of parameters), and re-run these same 6 questions. Then we compare directly.